### **Student Information**
Name (optional): 范祥彥

Student ID (optional): 114065525

GitHub ID: EvilOrbsMan

Kaggle name: FanIan

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a backup ZIP on the corresponding NTU COOL assignment. The ZIP should contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

# Project Report

---

## 1. Model Development

### 1.1 Preprocessing Steps

To prepare the 4,800 competition posts (`3,360` training instances and `1,440` test instances across six Ekman emotion categories: `joy`, `anger`, `sadness`, `disgust`, `surprise`, and `fear`), we implemented a structured data integration and text normalization pipeline:

- **Dataset Merging & Alignment**:
  - Loaded the raw text records from `final_posts.json` and merged them with `data_identification.csv` (`train` vs. `test` split) and `emotion.csv` (ground-truth training labels) using `post_id` as the primary key.
  - Strictly aligned the test set ordering (`1,440` rows) with `sampleSubmission.csv` via a left join on `post_id` to prevent any row misalignment during submission generation.
- **Emotion-Aware Text Normalization (`clean_post`)**:
  - **URL Removal**: Stripped web links (`http...`, `www...`) as they introduce unique token noise without generalizable emotional signal.
  - **User Mention Normalization**: Replaced specific user handles (`@username`) with a canonical `@user` token to prevent overfitting to individual accounts while preserving conversational structure.
  - **Hashtag Text Retention**: Removed the `#` prefix while preserving the underlying word (`#(\w+) -> \1`), since hashtags in social media posts frequently carry explicit emotional keywords.
  - **Punctuation & Case Normalization**: Normalized whitespace and lowercased text for lexical vectorizers while preserving emotion-carrying punctuation (such as `!` and `?`) and contractions (`don't`, `can't`), while passing the original uncleaned text to the pre-trained language model (`all-MiniLM-L6-v2`) to preserve case and punctuation context.

---

### 1.2 Feature Engineering Steps

Relying solely on word-level bag-of-words features struggles with informal spelling, morphological variations, and paraphrased expressions without exact keyword overlap. We therefore engineered a **Hybrid Lexical + Semantic Representation (`26,995` total features)** combining three complementary views of each post:

- **Word-Level TF-IDF (`7,439` features)**:
  - Extracted word unigrams and bigrams (`ngram_range=(1, 2)`, `min_df=2`, `max_df=0.95`, `sublinear_tf=True`).
  - Bigrams capture critical local modifiers and negations (e.g., `"not happy"`, `"so excited"`), while sublinear term-frequency scaling dampens repetitive words.
- **Character N-gram TF-IDF (`19,172` features)**:
  - Extracted character n-grams within word boundaries (`analyzer="char_wb"`, `ngram_range=(3, 5)`, `min_df=3`, `max_df=0.95`, `sublinear_tf=True`).
  - Captures subword morphology (prefixes, suffixes like `-ing`, `-ed`), expressive punctuation patterns (`!!!`, `?!`), and informal spelling variations common in short social posts.
- **Dense Semantic Embeddings via Pre-trained MiniLM (`384` features)**:
  - Encoded each post using `SentenceTransformer("all-MiniLM-L6-v2")` and applied L2 normalization (`normalize`), scaled by a factor of `1.5` before horizontal sparse concatenation (`scipy.sparse.hstack`).
  - Provides dense semantic representations that group paraphrased emotional expressions even when they share zero surface words with the training vocabulary.

---

### 1.3 Explanation of Your Model

Noticeable class imbalance exists in the training set (`joy`: 875, `anger`: 674, `sadness`: 628, `disgust`: 454, `surprise`: 441, `fear`: 288). Because the competition evaluates macro-averaged F1 across classes, a standard classifier without class weighting tends to under-predict minority classes such as `fear` and `surprise`.

- **Classifier Architecture & Regularization**:
  - We compared **Linear Support Vector Classification (`LinearSVC`)** and **Multinomial Logistic Regression (`LogisticRegression`)** across feature sets using **5-Fold Stratified Cross-Validation** (`random_state=42`).
  - We enabled **`class_weight="balanced"`** across all candidate models to weight each class inversely proportional to its frequency, protecting minority classes (`fear`, `surprise`, `disgust`).
  - For our high-dimensional hybrid feature space (`26,995` dimensions), stronger L2 regularization (`C=0.2` in `LinearSVC`) prevented overfitting to noisy lexical n-grams while leveraging the dense semantic dimensions.
- **Empirical Ablation Results (5-Fold Stratified CV)**:

| Model & Feature Configuration | 5-Fold CV Macro F1 | 5-Fold CV Weighted F1 |
| :--- | :---: | :---: |
| Word TF-IDF (1–2) + `LinearSVC(C=0.5)` | 0.4806 | 0.4981 |
| Word + Char TF-IDF + `LinearSVC(C=0.3)` | 0.5046 | 0.5225 |
| MiniLM Embeddings + `LinearSVC(C=0.5)` | 0.5024 | 0.5255 |
| MiniLM Embeddings + `LogisticRegression(C=2.0)` | 0.4982 | 0.5197 |
| **Hybrid (Word + Char TF-IDF + MiniLM) + `LinearSVC(C=0.2)`** | **0.5442** | **0.5634** |
| Hybrid (Word + Char TF-IDF + MiniLM) + `LogisticRegression(C=1.5)` | 0.5396 | 0.5590 |

- **Validation & Leaderboard Outcome**:
  - While plain Word TF-IDF scored `0.4806` (below the `0.50` baseline), combining Word TF-IDF, Character N-grams, and MiniLM embeddings boosted 5-Fold CV Macro F1 by **+6.36 percentage points** to **`0.5442`**, achieving **`0.5353`** on the Kaggle Public Leaderboard.

---

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

During development, we collaborated with an AI assistant (Gemini / Claude) to design and debug our pipeline. Below is the record of our prompts, the AI's suggestions, and our decisions:

1. **Prompt 1 — Dataset Schema Inspection & Pipeline Design**:
   - *Prompt*: Provided the directory file listing (`data_identification.csv`, `emotion.csv`, `final_posts.json`, `sampleSubmission.csv`) and asked how to inspect the JSON schema, merge splits safely, and build a baseline exceeding $F_1 \ge 0.50$.
   - *AI Suggestion*: Suggested writing an inspection script to verify the JSON keys (`post_id`, `text`), check class balance across the 6 emotion labels, and align `test_df` directly with `sampleSubmission.csv` order via a left join before feature extraction.
   - *Decision & Rationale*: **Adopted.** Inspection revealed 3,360 training and 1,440 test posts with an imbalanced 6-class distribution (`joy` at 875 down to `fear` at 288). Aligning `test_df` via left merge on `sampleSubmission["post_id"]` guaranteed zero ordering mismatches.
2. **Prompt 2 — Feature Engineering & Hybrid Lexical-Semantic Fusion**:
   - *Prompt*: Asked how to construct a CPU-efficient feature pipeline that outperforms a standard Word TF-IDF + SVM baseline on short social media posts.
   - *AI Suggestion*: Recommended combining (a) Word unigram/bigram TF-IDF with sublinear scaling, (b) Character boundary n-grams (`char_wb`, 3–5 chars) for subword and punctuation cues, and (c) L2-normalized `all-MiniLM-L6-v2` sentence embeddings via `scipy.sparse.hstack`, paired with `LinearSVC(C=0.2, class_weight="balanced")`.
   - *Decision & Rationale*: **Adopted the Hybrid TF-IDF + MiniLM + Balanced LinearSVC pipeline.** Our 5-fold CV ablation confirmed that Word TF-IDF alone only reached `0.4806` Macro F1, whereas the Hybrid fusion jumped to `0.5442` in under 25 seconds on CPU.
3. **Prompt 3 — Debugging Kaggle Submission Header Mismatch (`post_id` vs. `id`)**:
   - *Prompt*: Shared the Kaggle submission error (`ID column id not found in submission`) after uploading the first CSV generated using `sampleSubmission.csv`'s original header (`post_id,label`).
   - *AI Suggestion*: Identified that while `sampleSubmission.csv` listed `post_id` as the column name, Kaggle's evaluation parser required `id` as the primary key header, and suggested changing the output DataFrame dictionary to `{"id": test_df["post_id"], "label": test_preds}`.
   - *Decision & Rationale*: **Adopted.** Renaming the first column from `post_id` to `id` resolved the submission error immediately and achieved **`0.5353`** on the Public Leaderboard.

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [3]:
import json
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report

# 1. Load raw competition files from data/kaggle
kaggle_dir = "data/kaggle"
with open(f"{kaggle_dir}/final_posts.json", "r", encoding="utf-8") as f:
    posts_df = pd.DataFrame(json.load(f))

id_df = pd.read_csv(f"{kaggle_dir}/data_identification.csv")
emo_df = pd.read_csv(f"{kaggle_dir}/emotion.csv")
sub_df = pd.read_csv(f"{kaggle_dir}/sampleSubmission.csv")

# 2. Merge posts with split identification and training labels
df = posts_df.merge(id_df, on="post_id", how="left").merge(emo_df, on="post_id", how="left")

# 3. Text cleaning function preserving emotion-relevant punctuation (!, ?) and contractions
def clean_post(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)       # Remove URLs
    text = re.sub(r"@\w+", "@user", text)              # Normalize user mentions
    text = re.sub(r"#(\w+)", r"\1", text)              # Keep hashtag word, strip '#' symbol
    text = re.sub(r"\s+", " ", text).strip().lower()   # Normalize whitespace and lowercase
    return text

df["clean_text"] = df["text"].apply(clean_post)

# 4. Separate train (3,360 rows) and test (1,440 rows, ordered exactly like sampleSubmission.csv)
train_df = df[df["set"] == "train"].reset_index(drop=True)
test_df = sub_df[["post_id"]].merge(df[["post_id", "text", "clean_text"]], on="post_id", how="left")

print(f"Train shape: {train_df.shape}, Missing texts: {train_df['clean_text'].isna().sum()}")
print(f"Test shape : {test_df.shape}, Missing texts: {test_df['clean_text'].isna().sum()}")
print("\nSample cleaned texts:")
print(train_df[["post_id", "label", "clean_text"]].head(3))

Train shape: (3360, 5), Missing texts: 0
Test shape : (1440, 3), Missing texts: 0

Sample cleaned texts:
   post_id    label                        clean_text
0  f8bv5zh  disgust  thought it looked weird as well.
1  q0rr1l9  disgust                bear died in a pit
2  p94rwgk  sadness             i have endured worse.


## 2. Feature Engineering Steps

In [4]:
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

# 1. Word-level TF-IDF (captures unigrams and bigrams with sublinear TF scaling)
word_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)
X_train_word = word_tfidf.fit_transform(train_df["clean_text"])
X_test_word = word_tfidf.transform(test_df["clean_text"])

# 2. Character-level TF-IDF (captures subword morphology, suffixes, and punctuation emphasis)
char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
)
X_train_char = char_tfidf.fit_transform(train_df["clean_text"])
X_test_char = char_tfidf.transform(test_df["clean_text"])

# 3. Pre-trained MiniLM dense semantic embeddings (384 dimensions, L2-normalized)
sbert = SentenceTransformer("all-MiniLM-L6-v2")
X_train_sbert = normalize(sbert.encode(train_df["text"].tolist(), show_progress_bar=True))
X_test_sbert = normalize(sbert.encode(test_df["text"].tolist(), show_progress_bar=True))

# 4. Combine lexical TF-IDF features and semantic embeddings into a unified feature matrix
X_train_tfidf_all = hstack([X_train_word, X_train_char])
X_test_tfidf_all = hstack([X_test_word, X_test_char])

X_train_hybrid = hstack([X_train_word, X_train_char, csr_matrix(X_train_sbert * 1.5)])
X_test_hybrid = hstack([X_test_word, X_test_char, csr_matrix(X_test_sbert * 1.5)])

y_train = train_df["label"].values

print(f"Word TF-IDF shape  : {X_train_word.shape}")
print(f"Char TF-IDF shape  : {X_train_char.shape}")
print(f"MiniLM shape       : {X_train_sbert.shape}")
print(f"Hybrid matrix shape: {X_train_hybrid.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/105 [00:00<?, ?it/s]

Batches:   0%|          | 0/45 [00:00<?, ?it/s]

Word TF-IDF shape  : (3360, 7439)
Char TF-IDF shape  : (3360, 19172)
MiniLM shape       : (3360, 384)
Hybrid matrix shape: (3360, 26995)


## 3. Model Implementation Steps

In [7]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Evaluate candidate representations and classifiers using 5-fold Stratified CV (Macro F1)
candidates = [
    ("Word TF-IDF + LinearSVC (C=0.5)", X_train_word, X_test_word,
     LinearSVC(C=0.5, class_weight="balanced", random_state=42)),
    ("Word+Char TF-IDF + LinearSVC (C=0.3)", X_train_tfidf_all, X_test_tfidf_all,
     LinearSVC(C=0.3, class_weight="balanced", random_state=42)),
    ("MiniLM + LinearSVC (C=0.5)", X_train_sbert, X_test_sbert,
     LinearSVC(C=0.5, class_weight="balanced", random_state=42)),
    ("MiniLM + LogisticRegression (C=2.0)", X_train_sbert, X_test_sbert,
     LogisticRegression(C=2.0, class_weight="balanced", max_iter=1000, random_state=42)),
    ("Hybrid (TF-IDF + MiniLM) + LinearSVC (C=0.2)", X_train_hybrid, X_test_hybrid,
     LinearSVC(C=0.2, class_weight="balanced", random_state=42)),
    ("Hybrid (TF-IDF + MiniLM) + LogReg (C=1.5)", X_train_hybrid, X_test_hybrid,
     LogisticRegression(C=1.5, class_weight="balanced", max_iter=1000, random_state=42)),
]

best_score = -1.0
best_name = None
best_model = None
best_X_tr, best_X_te = None, None

print("=== 5-Fold Stratified Cross-Validation Results ===")
for name, X_tr, X_te, clf in candidates:
    f1_macro = cross_val_score(clf, X_tr, y_train, cv=skf, scoring="f1_macro", n_jobs=-1).mean()
    f1_weighted = cross_val_score(clf, X_tr, y_train, cv=skf, scoring="f1_weighted", n_jobs=-1).mean()
    print(f"{name:45s} | CV Macro F1: {f1_macro:.4f} | Weighted F1: {f1_weighted:.4f}")
    if f1_macro > best_score:
        best_score = f1_macro
        best_name = name
        best_model = clf
        best_X_tr, best_X_te = X_tr, X_te

print(f"\nSelected Best Model: {best_name} (CV Macro F1 = {best_score:.4f})")

# Fit the winning model on the full training set and generate predictions
best_model.fit(best_X_tr, y_train)
test_preds = best_model.predict(best_X_te)

submission = pd.DataFrame({
    "id": test_df["post_id"],
    "label": test_preds,
})
submission.to_csv("submission_baseline.csv", index=False)

print(f"\nSaved 'submission_baseline.csv' with shape {submission.shape}")
print("Predicted test label distribution:")
print(submission["label"].value_counts())
print("\nFirst 5 rows of submission_baseline.csv:")
print(submission.head())

=== 5-Fold Stratified Cross-Validation Results ===
Word TF-IDF + LinearSVC (C=0.5)               | CV Macro F1: 0.4806 | Weighted F1: 0.4981
Word+Char TF-IDF + LinearSVC (C=0.3)          | CV Macro F1: 0.5046 | Weighted F1: 0.5225
MiniLM + LinearSVC (C=0.5)                    | CV Macro F1: 0.5024 | Weighted F1: 0.5255
MiniLM + LogisticRegression (C=2.0)           | CV Macro F1: 0.4982 | Weighted F1: 0.5197
Hybrid (TF-IDF + MiniLM) + LinearSVC (C=0.2)  | CV Macro F1: 0.5442 | Weighted F1: 0.5634
Hybrid (TF-IDF + MiniLM) + LogReg (C=1.5)     | CV Macro F1: 0.5396 | Weighted F1: 0.5590

Selected Best Model: Hybrid (TF-IDF + MiniLM) + LinearSVC (C=0.2) (CV Macro F1 = 0.5442)

Saved 'submission_baseline.csv' with shape (1440, 2)
Predicted test label distribution:
label
joy         399
anger       309
sadness     258
surprise    174
disgust     167
fear        133
Name: count, dtype: int64

First 5 rows of submission_baseline.csv:
        id     label
0  xyzx9jp     anger
1  l3vzzxt      fe